# LLM-Based Term Extraction (Zero-Shot, Local Models)

This notebook runs a small instruction-tuned model directly on the Colab GPU:
- No Groq/OpenAI API calls or API keys; model weights are downloaded from Hugging Face once per fresh runtime
- Keeps the original zero-shot prompt, document batching, output format, saving, and evaluation
- No training or fine-tuning required

### Run on free Google Colab
1. Upload/open this notebook and select **Runtime > Change runtime type > T4 GPU** (or another available GPU).
2. Make the dataset available in the Colab runtime: upload `train.json` and `test.json` using the Files sidebar, or upload/clone the repository. Set `DATA_DIR` in the data-loading cell if necessary. Opening the notebook alone does not upload the dataset.
3. Run the cells from the top. If Colab asks for a session restart after installation, restart and rerun from the imports.
4. Start with `MAX_DOCUMENTS = 20` in the extraction section, then set it to `None` for the full test set.
5. Download saved predictions from the Files sidebar before the temporary runtime ends.

The default is `Qwen/Qwen2.5-1.5B-Instruct`; change one line to use `Qwen/Qwen2.5-0.5B-Instruct` for a smaller model. FP16 weights alone occupy approximately 3 GB / 1 GB respectively; runtime memory is higher. No quantization is needed for these models on a T4. Small models may have weaker biomedical extraction and formatting accuracy: compare them with the same evaluation below. Free Colab GPU access and session duration depend on availability and usage limits.

References: [1.5B model](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), [0.5B model](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct), [chat templates](https://huggingface.co/docs/transformers/v4.57.1/chat_templating), [Colab limits](https://research.google.com/colaboratory/faq.html).


## Setup and Imports

In [1]:
%pip install -q "transformers==4.57.1" "accelerate>=1.0,<2" bitsandbytes pandas tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import sys
import torch
import transformers
import bitsandbytes as bnb

from transformers.utils import is_bitsandbytes_available

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)
print("bitsandbytes location:", bnb.__file__)
print(
    "Detected by Transformers:",
    is_bitsandbytes_available(check_library_only=True),
)

assert torch.cuda.is_available(), "Enable a GPU runtime."
assert is_bitsandbytes_available(check_library_only=True), (
    "Transformers still cannot detect bitsandbytes."
)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130
PyTorch CUDA: 13.0
Transformers: 4.57.1
bitsandbytes: 0.50.2
bitsandbytes location: /usr/local/lib/python3.13/dist-packages/bitsandbytes/__init__.py
Detected by Transformers: True


In [3]:
import pandas as pd
import json
import os
import re
from tqdm import tqdm

print("Libraries imported")

Libraries imported


## Data Loading and Processing

In [4]:
# Set DATA_DIR
DATA_DIR = "/content/"

# Load training and test data
train_data = json.load(open(f"{DATA_DIR}/train.json", 'r', encoding='utf-8'))
test_data = json.load(open(f"{DATA_DIR}/test.json", 'r', encoding='utf-8'))

print(f"Training docs: {len(train_data)}")
print(f"Test docs: {len(test_data)}")
print(f"\nExample sentence:")
print(f"  Text: {test_data[list(test_data.keys())[0]]['text']}")
print(f"  Terms: {test_data[list(test_data.keys())[0]]['terms']}")


Training docs: 540
Test docs: 567

Example sentence:
  Text: Gut microbiota disturbances in hospitalized older adults with malnutrition and clinical outcomes.  OBJECTIVE: Malnutrition is one of the most threatening conditions in geriatric  populations. The gut microbiota has an important role in the host's metabolic and muscular health: however, its interplay with disease-related malnutrition is not well understood. We aimed to identify the association of malnutrition with the gut microbiota and predict clinical outcomes in hospitalized acutely ill older adults. METHODS: We performed a secondary longitudinal analysis in 108 geriatric patients from a prospective cohort evaluated at admission and 72 h of hospitalization. We collected clinical, demographic, nutritional, and 16S rRNA gene-sequenced gut microbiota data. Microbiota diversity, overall composition, and differential abundance were calculated and compared between patients with and without malnutrition. Microbiota features associ

## Evaluation Metrics

Using the official evaluation metrics from the competition.

In [5]:
def micro_f1_score(gold_standard, system_output):
    """
    Evaluates performance using Precision, Recall, and F1 score
    based on individual term matching (micro-average).
    """
    total_true_positives = 0
    total_false_positives = 0
    total_false_negatives = 0

    for gold, system in zip(gold_standard, system_output):
        gold_set = set(gold)
        system_set = set(system)

        true_positives = len(gold_set.intersection(system_set))
        false_positives = len(system_set - gold_set)
        false_negatives = len(gold_set - system_set)

        total_true_positives += true_positives
        total_false_positives += false_positives
        total_false_negatives += false_negatives

    precision = total_true_positives / (total_true_positives + total_false_positives) if (total_true_positives + total_false_positives) > 0 else 0
    recall = total_true_positives / (total_true_positives + total_false_negatives) if (total_true_positives + total_false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, total_true_positives, total_false_positives, total_false_negatives


def type_f1_score(gold_standard, system_output):
    """
    Evaluates performance using Type Precision, Type Recall, and Type F1 score
    based on the set of unique terms extracted at least once across the entire dataset.
    """
    all_gold_terms = set()
    for item_terms in gold_standard:
        all_gold_terms.update(item_terms)

    all_system_terms = set()
    for item_terms in system_output:
        all_system_terms.update(item_terms)

    type_true_positives = len(all_gold_terms.intersection(all_system_terms))
    type_false_positives = len(all_system_terms - all_gold_terms)
    type_false_negatives = len(all_gold_terms - all_system_terms)

    type_precision = type_true_positives / (type_true_positives + type_false_positives) if (type_true_positives + type_false_positives) > 0 else 0
    type_recall = type_true_positives / (type_true_positives + type_false_negatives) if (type_true_positives + type_false_negatives) > 0 else 0
    type_f1 = 2 * (type_precision * type_recall) / (type_precision + type_recall) if (type_precision + type_recall) > 0 else 0

    return type_precision, type_recall, type_f1


print("Evaluation functions defined")

Evaluation functions defined


## Initialize Local LLM Model

Load the model once and reuse it for every batch. `generate_response()` applies the model's chat template, runs greedy generation on the GPU, and decodes **only newly generated tokens**. It returns a lightweight object with the same fields used by the original notebook (`response.choices[0].message.content`, token usage, etc.), so the rest of the workflow needs very few changes.

`batch_size` below is the number of documents inside one prompt, not a tensor batch of independent prompts. `max_new_tokens` limits the generated answer, not the input. If a response has finish reason `length`, increase this limit or reduce the document batch size.


In [ ]:
# Get your HF Token here: https://huggingface.co/settings/tokens
import os
os.environ["HF_TOKEN"] = input("Insert your HF Token and press Enter: ")

In [43]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from types import SimpleNamespace

# FULL MODELS
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
# model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # Smaller alternative

# QUANTIZED MODELS
# model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit" # Smaller alternative

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Colab before loading the model.")

print(f"GPU: {torch.cuda.get_device_name(0)}")

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map={"": 0},        # Load directly onto the Colab GPU.
    torch_dtype=torch.float16,
)
model.eval()

print(f"Loaded {model_name} on {model.device}")
print(f"Model memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GiB")

GPU: Tesla T4


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-1.5B-Instruct on cuda:0
Model memory footprint: 2.88 GiB


In [44]:
@torch.inference_mode()
def generate_response(messages, max_new_tokens=512):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    prompt_tokens = inputs['input_ids'].shape[-1]
    if prompt_tokens + max_new_tokens > model.config.max_position_embeddings:
        raise ValueError("Prompt plus output budget exceeds the model context; reduce batch_size.")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # Greedy decoding, equivalent to temperature=0.
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_tokens = outputs[0, prompt_tokens:]
    content = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    completion_tokens = len(generated_tokens)
    eos_ids = model.generation_config.eos_token_id
    eos_ids = [eos_ids] if isinstance(eos_ids, int) else (eos_ids or [])
    stopped_on_eos = completion_tokens > 0 and generated_tokens[-1].item() in eos_ids
    finish_reason = "stop" if stopped_on_eos or completion_tokens < max_new_tokens else "length"
    if finish_reason == "length":
        print("Warning: output hit max_new_tokens; terms may be incomplete.")

    # Preserve the response fields used in the original API-based notebook.
    return SimpleNamespace(
        model=model_name,
        choices=[SimpleNamespace(
            message=SimpleNamespace(content=content),
            finish_reason=finish_reason,
        )],
        usage=SimpleNamespace(
            prompt_tokens=prompt_tokens,
            completion_tokens=completion_tokens,
            total_tokens=prompt_tokens + completion_tokens,
        ),
    )

print(f"Loaded {model_name} locally on {model.device}")


Loaded Qwen/Qwen2.5-1.5B-Instruct locally on cuda:0


In [45]:
response = generate_response(
    messages=[{"role": "user", "content": "Hi, how are you?"}],
    max_new_tokens=64,
)
print(response)


namespace(model='Qwen/Qwen2.5-1.5B-Instruct', choices=[namespace(message=namespace(content="Hello! I'm just an AI assistant here to help with your questions and provide information. How can I assist you today?"), finish_reason='stop')], usage=namespace(prompt_tokens=35, completion_tokens=26, total_tokens=61))


In [46]:
print(response.choices[0].message.content)


Hello! I'm just an AI assistant here to help with your questions and provide information. How can I assist you today?


In [47]:
print(response.usage.prompt_tokens)
print(response.usage.completion_tokens)
print(response.usage.total_tokens)

35
26
61


## Configure Zero-Shot Prompt

In [52]:
# Configure batch size for efficient processing
batch_size = 1

system_prompt = f"""You are an automatic term extraction agent. You will receive a list of texts as input.
Your role is to extract terms relevant to the gut-brain interplay field of research, including gut microbiota, mental health, and neurodegenerative diseases. Output a list of terms for each sentence.

Strictly adhere to the Example Output Format:

Example Output Format:
Document 1: [term1; term2; term3; term4]
Document 2: [term5; term6]
Document 3: []
Document 4: []
Document 5: [term7]

Instructions:
* Extract single-word and multi-word terms, including domain-relevant organism names such as Akkermansia muciniphila
* Extract domain-relevant lexical units even if they are not listed in biomedical dictionaries or ontologies
* Extract complete lexical units, including relevant modifiers, rather than their nested component terms
* Split coordinated expressions into separate terms, restoring shared words where needed: "cognitive or motor dysfunction" yields "cognitive dysfunction" and "motor dysfunction"
* Extract only terms relevant to the gut-brain interplay, ignoring other domains
* If a document contains no terms, output an empty list for that document
* Output exactly one list of terms for each input document, using its Document number, without additional commentary
"""

system_prompt_single_doc = f"""You are an automatic term extraction agent. You will receive a biomedical text as input.
Your role is to extract terms relevant to the gut-brain interplay field of research, including gut microbiota, mental health, and neurodegenerative diseases. Output a list of terms.

Strictly adhere to the Example Output Format:

Example Output Format:
Terms: [term1; term2; term3; term4]

Instructions:
* Extract single-word and multi-word terms, including domain-relevant organism names such as Akkermansia muciniphila
* Extract domain-relevant lexical units even if they are not listed in biomedical dictionaries or ontologies
* Extract complete lexical units, including relevant modifiers, rather than their nested component terms
* Split coordinated expressions into separate terms, restoring shared words where needed: "cognitive or motor dysfunction" yields "cognitive dysfunction" and "motor dysfunction"
* Extract only terms relevant to the gut-brain interplay, ignoring other domains
* If a document contains no terms, output an empty list for that document
* You must output a single list of terms without additional commentary
"""

if batch_size == 1:
  system_prompt = system_prompt_single_doc

print(f"Zero-shot prompt configured (batch size: {batch_size})")

Zero-shot prompt configured (batch size: 1)


## Extract Terms Using LLM

In [72]:
test_data = json.load(open(f"{DATA_DIR}/test.json", 'r', encoding='utf-8'))
MAX_DOCUMENTS = 20  # Quick first run; use None to evaluate the full test set.
if MAX_DOCUMENTS is not None:
    test_data = dict(list(test_data.items())[:MAX_DOCUMENTS])
print(f"Selected {len(test_data)} test documents")


Selected 20 test documents


In [73]:
# Process documents in batches with LLM
print(f"Processing {len(test_data)} documents in batches of {batch_size}...")

raw_responses = []
response_list = []
user_prompt = ""

i=0
for id, content in tqdm(test_data.items(), desc="Processing documents..."):
    document = content['text']
    i += 1

    # Build prompt until batch size is reached
    if i % batch_size == 0:
        user_prompt += f"Document {i}:\n {document}\n\n"

        # Send batch to LLM
        response = generate_response(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt + f"\nReturn exactly {i % batch_size or batch_size} lists, using the Document numbers above."}
            ],
            max_new_tokens=4096,
        )

        raw_responses.append(response)
        response_list.append(response.choices[0].message.content)

        user_prompt = "" # Reset user prompt for next batch
    else:
        user_prompt += f"Document {i}:\n {document}\n\n"

# Process remaining documents (last batch)
if user_prompt:
    user_prompt = user_prompt.rstrip()
    response = generate_response(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + f"\nReturn exactly {i % batch_size or batch_size} lists, using the Document numbers above."}
        ],
        max_new_tokens=4096,
    )
    raw_responses.append(response)
    response_list.append(response.choices[0].message.content)


print(f"Received {len(response_list)} batch responses from LLM")

Processing 20 documents in batches of 1...


Processing documents...:  15%|█▌        | 3/20 [02:23<18:09, 64.06s/it]

Processing documents...:  55%|█████▌    | 11/20 [04:54<06:55, 46.12s/it]

Processing documents...: 100%|██████████| 20/20 [05:19<00:00, 15.98s/it]

Received 20 batch responses from LLM


### Analyze LLM outputs

In [74]:
for batch_num, response in enumerate(raw_responses, start=1):
    choice = response.choices[0]

    print(f"\n--- Batch {batch_num} ---")
    print("Model:", response.model)
    print("Finish reason:", choice.finish_reason)
    print("Content:\n", choice.message.content)
    print("Usage:", response.usage)

    if not (choice.message.content or "").strip():
        print("Full message:", vars(choice.message))


--- Batch 1 ---
Model: Qwen/Qwen2.5-1.5B-Instruct
Finish reason: stop
Content:
 [Association; Malnutrition; Geriatric; Gut microbiota; Clinical outcome; Acute illness; Hospitalization; Nutritional status; Dysbiosis; Alpha diversity; Beta diversity; Shannon index; Simpson index; Chao1 index; Critical care; Microbiota feature; Differential abundance; Corynebacterium; Ruminococcaceae Incertae Sedis; Fusobacterium; Lachnospiraceae NK4A136 group; Subdoligranulum; Faecalibacterium prausnitzii]
Usage: namespace(prompt_tokens=719, completion_tokens=116, total_tokens=835)

--- Batch 2 ---
Model: Qwen/Qwen2.5-1.5B-Instruct
Finish reason: stop
Content:
 [Preface; Special issue; Ion channels; Genetic epilepsy; Recurrent seizures; Standard of care treatments; Pathogenic mechanisms; Genes; Ions; Channels; HCN channels; Voltage-dependent sodium channels; Voltage-dependent calcium channels; GABAA receptors; Review articles; Precision medicine; Metabolic aspects; Ion channels; NMDA receptor; SCN2A dis

In [75]:
for response in response_list:
    print(f"Response:\n{response}\n{'-'*40}")

Response:
[Association; Malnutrition; Geriatric; Gut microbiota; Clinical outcome; Acute illness; Hospitalization; Nutritional status; Dysbiosis; Alpha diversity; Beta diversity; Shannon index; Simpson index; Chao1 index; Critical care; Microbiota feature; Differential abundance; Corynebacterium; Ruminococcaceae Incertae Sedis; Fusobacterium; Lachnospiraceae NK4A136 group; Subdoligranulum; Faecalibacterium prausnitzii]
----------------------------------------
Response:
[Preface; Special issue; Ion channels; Genetic epilepsy; Recurrent seizures; Standard of care treatments; Pathogenic mechanisms; Genes; Ions; Channels; HCN channels; Voltage-dependent sodium channels; Voltage-dependent calcium channels; GABAA receptors; Review articles; Precision medicine; Metabolic aspects; Ion channels; NMDA receptor; SCN2A disease; Primary data manuscripts; Gut microbiota; Anti-seizure drugs; In vivo screening tools]
----------------------------------------
Response:
[Association; gut microbiota dysbi

## Parse LLM Responses

In [76]:
# Parse by document number so a missing/malformed line cannot shift later predictions.
print("Parsing LLM responses...")

llm_preds = [[] for _ in test_data]
if batch_size == 1:
  idx = 0
  for id, content in test_data.items():
      if idx > len(response_list) - 1:
          break
      llm_preds[idx] = [term.strip().lower() for term in response_list[idx].split(';') if term.strip()]
      idx += 1
else:
  for batch_index, response in enumerate(response_list):
      first_id = batch_index * batch_size + 1
      expected_ids = set(range(first_id, min(first_id + batch_size, len(test_data) + 1)))
      seen_ids = set()
      for line in response.splitlines():
          match = re.fullmatch(r'\s*Document\s+(\d+):\s*\[(.*?)\]\s*', line)
          if not match:
              continue
          doc_num = int(match.group(1))
          if doc_num not in expected_ids or doc_num in seen_ids:
              print(f"Warning: ignoring unexpected/duplicate Document {doc_num} in batch {batch_index + 1}")
              continue
          terms = [term.strip().lower() for term in match.group(2).split(';') if term.strip()]
          llm_preds[doc_num - 1] = terms
          seen_ids.add(doc_num)
      missing_ids = expected_ids - seen_ids
      if missing_ids:
          print(f"Warning: missing/malformed output for Documents {sorted(missing_ids)}; using empty lists.")

print(f"Parsed {len(llm_preds)} predictions")


Parsing LLM responses...
Parsed 20 predictions


In [77]:
import re

print("Parsing LLM responses...")

header_pattern = re.compile(
    r"\bDocument\s+(\d+)\s*\*{0,2}\s*:",
    flags=re.IGNORECASE,
)


def parse_terms(text):
    """Return (terms, truncated), or None if no bracketed list exists."""
    start = text.find("[")
    if start == -1:
        return None

    body = text[start + 1:]
    end = body.find("]")
    truncated = end == -1

    if truncated:
        # The last fragment may be an unfinished term.
        body = body.rsplit(";", 1)[0] if ";" in body else ""
    else:
        body = body[:end]

    terms = []
    seen = set()
    for fragment in body.split(";"):
        term = fragment.strip().strip("\"'").strip().lower()
        if term and term not in seen:
            terms.append(term)
            seen.add(term)

    return terms, truncated

if batch_size > 1:
  llm_preds = [[] for _ in test_data]
  expected_batches = (len(test_data) + batch_size - 1) // batch_size

  if len(response_list) != expected_batches:
      print(
          f"Warning: expected {expected_batches} batch responses, "
          f"received {len(response_list)}."
      )

  for batch_index, response in enumerate(response_list[:expected_batches]):
      first_id = batch_index * batch_size + 1
      expected_ids = list(
          range(first_id, min(first_id + batch_size, len(test_data) + 1))
      )
      text = response or ""
      headers = list(header_pattern.finditer(text))
      entries = []

      for index, header in enumerate(headers):
          end = headers[index + 1].start() if index + 1 < len(headers) else len(text)
          parsed = parse_terms(text[header.end():end])
          if parsed is not None:
              entries.append((int(header.group(1)), *parsed))

      # For one input document, its batch position determines its identity.
      # Accept a wrong label (e.g. "Document 1" for the third document).
      if len(expected_ids) == 1:
          if not headers:
              parsed = parse_terms(text)
              if parsed is not None:
                  entries = [(expected_ids[0], *parsed)]

          if len(entries) == 1:
              label, terms, truncated = entries[0]
              doc_id = expected_ids[0]
              llm_preds[doc_id - 1] = terms

              if label != doc_id:
                  print(f"Warning: treating Document {label} as Document {doc_id}.")
              if truncated:
                  print(
                      f"Warning: Document {doc_id} was truncated; "
                      "kept complete terms only."
                  )
          else:
              print(
                  f"Warning: missing or ambiguous output for Document "
                  f"{expected_ids[0]}; using an empty list."
              )
          continue

      # Multiple input documents: accept global numbering or batch-local numbering.
      # Inspect all headers, including those with malformed lists.
      labels = [int(header.group(1)) for header in headers]
      local_ids = set(range(1, len(expected_ids) + 1))

      if labels and all(label in expected_ids for label in labels):
          mapping = {label: label for label in expected_ids}
      elif labels and all(label in local_ids for label in labels):
          mapping = {
              local_id: doc_id
              for local_id, doc_id in enumerate(expected_ids, start=1)
          }
      else:
          # Mixed/invalid numbering: retain only explicit global matches.
          mapping = {label: label for label in expected_ids}
          print(f"Warning: inconsistent numbering in batch {batch_index + 1}.")

      seen_ids = set()
      for label, terms, truncated in entries:
          doc_id = mapping.get(label)
          if doc_id is None or doc_id in seen_ids:
              print(f"Warning: ignoring unexpected/duplicate Document {label}.")
              continue

          llm_preds[doc_id - 1] = terms
          seen_ids.add(doc_id)

          if truncated:
              print(
                  f"Warning: Document {doc_id} was truncated; "
                  "kept complete terms only."
              )

      missing_ids = set(expected_ids) - seen_ids
      if missing_ids:
          print(
              f"Warning: missing/malformed output for Documents "
              f"{sorted(missing_ids)}; using empty lists."
          )

  print(f"Parsed {len(llm_preds)} predictions")

Parsing LLM responses...


## Save Predictions to File

In [78]:
def save_predictions(predictions, documents, output_path):
    """Save predictions in competition format."""
    output = {}
    idx = 0
    for id, content in documents.items():
        output[id] = {
            'text': content['text'],
            'terms': predictions[idx]
        }
        idx += 1

    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(predictions)} predictions to {output_path}")

save_predictions(llm_preds, test_data, f'predictions/subtaskA_{model_name.split("/")[-1].lower()}_preds.json')

Saved 20 predictions to predictions/subtaskA_qwen2.5-1.5b-instruct_preds.json


## Evaluate Performance

In [79]:
# Prepare gold standard and predictions for evaluation
test_gold = [content['terms'] for _, content in test_data.items()]

# Calculate metrics
precision, recall, f1, tp, fp, fn = micro_f1_score(test_gold, llm_preds)
type_precision, type_recall, type_f1 = type_f1_score(test_gold, llm_preds)

print("\n" + "="*60)
print("LLM ZERO-SHOT BASELINE RESULTS")
print("="*60)
print("\nMicro-averaged Metrics:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  TP={tp}, FP={fp}, FN={fn}")

print("\nType-level Metrics:")
print(f"  Type Precision: {type_precision:.4f}")
print(f"  Type Recall:    {type_recall:.4f}")
print(f"  Type F1 Score:  {type_f1:.4f}")
print("="*60)


LLM ZERO-SHOT BASELINE RESULTS

Micro-averaged Metrics:
  Precision: 0.2415
  Recall:    0.1440
  F1 Score:  0.1804
  TP=92, FP=289, FN=547

Type-level Metrics:
  Type Precision: 0.2315
  Type Recall:    0.1515
  Type F1 Score:  0.1831


In [80]:
## Example Predictions

In [81]:
predictions = json.load(open(f'predictions/subtaskA_{model_name.split("/")[-1].lower()}_preds.json', 'r', encoding='utf-8'))

# Show example predictions
print("Example Predictions:\n")

count = 0

for id, content in test_data.items():
    if len(content['terms']) > 0 and count < 5:
        print(f"Document ID: {id}")
        print(f"Text: {content['text'][:100]}...")
        print(f"Gold terms: {content['terms'][:5]}")
        print(f"LLM predictions: {predictions[id]['terms'][:5]}")

        correct = set(content['terms']) & set(predictions[id]['terms'])
        missed = set(content['terms']) - set(predictions[id]['terms'])
        wrong = set(predictions[id]['terms']) - set(content['terms'])

        print(f"Correct: {len(correct)}")
        print(f"Missed: {len(missed)}")
        print(f"Wrong: {len(wrong)}")
        print("-"*80)
        print()

        count += 1


Example Predictions:

Document ID: mh_2
Text: Gut microbiota disturbances in hospitalized older adults with malnutrition and clinical outcomes.  O...
Gold terms: ['Gut microbiota disturbances', 'malnutrition', 'clinical outcomes', 'Malnutrition', 'gut microbiota']
LLM predictions: ['[association', 'malnutrition', 'geriatric', 'gut microbiota', 'clinical outcome']
Correct: 4
Missed: 29
Wrong: 19
--------------------------------------------------------------------------------

Document ID: mh_3
Text: Preface: Special issue: "Ion channels and genetic epilepsy".  This preface introduces the Journal of...
Gold terms: ['Ion channels', 'genetic epilepsy', 'Epilepsy', 'disease', 'recurrent seizures']
LLM predictions: ['[preface', 'special issue', 'ion channels', 'genetic epilepsy', 'recurrent seizures']
Correct: 9
Missed: 21
Wrong: 14
--------------------------------------------------------------------------------

Document ID: mh_5
Text: Association between gut microbiota dysbiosis and poor f